# 20 — Selective MMR/GTE gold-type oracle

이 노트북은 **Codex coder agent**가 `skn25` CPU/offline fresh kernel에서 실행한 사후 gold query-type oracle 진단이다. 19번 결과를 본 뒤 Top50 λ=0.7을 primary로 정한 adaptive exploratory 분석이므로 promotion 대상이 아니며 운영 가능한 router가 아니다.

`selective_mmr_gold_type_oracle`은 numeric만 같은 depth/λ의 19 MMR을 선택하고 semantic/proper noun은 19 no-MMR을 선택한다. `combined_mmr_gte_gold_type_oracle`은 numeric은 19 MMR, semantic은 같은 0.4/depth의 18 GTE, proper noun은 19 no-MMR을 선택한다. 저장 metric/Top5 행을 정확히 복사할 뿐 새 ranking, score 혼합, 검색 또는 평가 계약 변경은 하지 않는다.


In [1]:
from pathlib import Path
import csv, hashlib, itertools, json, math, os, statistics, tempfile

PROJECT_ROOT=Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd(); assert (PROJECT_ROOT/'notebooks').is_dir()
OUTPUT_ROOT=PROJECT_ROOT/'notebooks/data/20_selective_mmr_gte_gold_type_oracle'; OUTPUT_ROOT.mkdir(parents=True,exist_ok=True)
SOURCE_13=PROJECT_ROOT/'notebooks/data/13_hierarchical_chunking_retrieval'; SOURCE_18=PROJECT_ROOT/'notebooks/data/18_answer_bearing_grouped_retrieval_reevaluation'; SOURCE_19=PROJECT_ROOT/'notebooks/data/19_mmr_redundancy_ablation'
INPUTS={
 'queries_13':SOURCE_13/'retrieval_per_query.csv', 'exact_groups_18':SOURCE_18/'exact_document_groups.jsonl', 'per_query_18':SOURCE_18/'per_query_metrics.csv', 'summary_18':SOURCE_18/'summary.csv', 'selective_gte_per_query_18':SOURCE_18/'selective_oracle_per_query.csv', 'selective_gte_summary_18':SOURCE_18/'selective_oracle_summary.csv', 'integrity_18':SOURCE_18/'selective_oracle_integrity.json',
 'per_query_19':SOURCE_19/'mmr_per_query_metrics.csv', 'summary_19':SOURCE_19/'mmr_summary.csv', 'redundancy_19':SOURCE_19/'mmr_redundancy_per_query.csv', 'trace_19':SOURCE_19/'mmr_step_trace.csv', 'decision_19':SOURCE_19/'mmr_selection_decision.json', 'integrity_19':SOURCE_19/'mmr_integrity.json',
}
DEPTHS=(20,50); LAMBDAS=(0.7,0.8,0.9); PRIMARY_DEPTH=50; PRIMARY_LAMBDA=0.7; TOLERANCE=1e-12
ORACLES=('selective_mmr_gold_type_oracle','combined_mmr_gte_gold_type_oracle')
VIEWS=('strict_raw','answer_bearing_raw','strict_exact_doc_dedup','answer_bearing_exact_doc_dedup','answer_bearing_gold_family_oracle'); METRICS=('card_hit_at_3','hit_at_3','recall_at_5','mrr_at_5','ndcg_at_5')
GROUPS=('all','card','evidence','numeric','semantic'); DENOMINATORS={'all':30,'card':10,'evidence':20,'numeric':10,'semantic':10}
REFERENCES={'selective_mmr_gold_type_oracle':('no_mmr','same_lambda_all_mmr'),'combined_mmr_gte_gold_type_oracle':('no_mmr','selective_gte_gold_type_oracle','same_lambda_selective_mmr','all_gte','same_lambda_all_mmr')}
SOURCE_PAYLOAD_FIELDS=('card_hit_at_3','hit_at_3','recall_at_5','mrr_at_5','ndcg_at_5','relevant_unit_count','ranking_unit_count','top5_chunk_ids','top5_cards','top5_levels')
CONTRACT={'schema_version':'selective_mmr_gte_gold_type_oracle_v2_canonical_source_keys','provenance':'Codex coder agent','scope':{'development_only':True,'post_hoc_gold_type_oracle':True,'adaptive_after_19_results':True,'primary':{'depth':50,'lambda':0.7},'promotion_eligible':False,'operational_router':False},'oracles':{'selective_mmr_gold_type_oracle':{'numeric_condition':'19 same-depth/lambda MMR','semantic':'19 no-MMR','proper_noun':'19 no-MMR'},'combined_mmr_gte_gold_type_oracle':{'numeric_condition':'19 same-depth/lambda MMR','semantic':'18 same-depth GTE','proper_noun':'19 no-MMR'}},'configs':{'weight':'vector_0.4_bm25_0.6','depths':list(DEPTHS),'lambdas':list(LAMBDAS),'views':list(VIEWS)},'method':'exact source metric/top5 row selection only; no new ranking, score mixing, retrieval, or evaluation change','source_key_contract':{'18':['comparison','weight','depth','system','query_id','view'],'19':['configuration','depth','system','query_id','view'],'ambiguous_18_key_without_comparison_weight':'prohibited','source_payload_digest_fields':list(SOURCE_PAYLOAD_FIELDS)},'comparisons':REFERENCES,'diagnostic_labels':{'support':'counterfactual_complementarity_supported_on_dev','tradeoff':'metric_tradeoff_on_dev','no_complementarity':'no_counterfactual_complementarity'},'final_status':['exploratory_gold_type_oracle_only','not_eligible_for_promotion','not_an_operational_router'],'execution':{'environment':'skn25','fresh_kernel':True,'cpu_only':True,'gpu_calls':0,'model_custom_code_calls':0,'network_api_calls':0,'new_embedding_calls':0,'chroma_queries':0,'package_installs':0}}
assert os.environ.get('CUDA_VISIBLE_DEVICES','')=='' and os.environ.get('HF_HUB_OFFLINE')=='1' and os.environ.get('TRANSFORMERS_OFFLINE')=='1'
def sha(path): return hashlib.sha256(path.read_bytes()).hexdigest()
def source_payload_digest(row): return hashlib.sha256(json.dumps({field:str(row[field]) for field in SOURCE_PAYLOAD_FIELDS},ensure_ascii=False,sort_keys=True,separators=(',',':')).encode()).hexdigest()
def read_csv(path): return list(csv.DictReader(path.open(encoding='utf-8',newline='')))
def read_jsonl(path): return [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines()]
def write_json(path,value):
    with tempfile.NamedTemporaryFile('w',encoding='utf-8',dir=path.parent,delete=False) as handle: temporary=Path(handle.name); json.dump(value,handle,ensure_ascii=False,indent=2); handle.write('\n')
    os.replace(temporary,path)
def write_csv(path,rows):
    rows=list(rows); columns=list(dict.fromkeys(key for row in rows for key in row))
    with tempfile.NamedTemporaryFile('w',encoding='utf-8',newline='',dir=path.parent,delete=False) as handle: temporary=Path(handle.name); writer=csv.DictWriter(handle,fieldnames=columns); writer.writeheader(); writer.writerows(rows)
    os.replace(temporary,path)
def in_group(row,group): return group=='all' or (group=='card' and row['question_group']=='card') or (group=='evidence' and row['question_group']=='evidence') or (group=='numeric' and row['category']=='numeric_condition') or (group=='semantic' and row['category']=='semantic')
input_hashes_before={name:sha(path) for name,path in INPUTS.items()}; write_json(OUTPUT_ROOT/'oracle_contract.json',CONTRACT)
print({'contract_frozen_before_results':True,'inputs':len(INPUTS),'oracles':ORACLES,'primary':[PRIMARY_DEPTH,PRIMARY_LAMBDA]})


{'contract_frozen_before_results': True, 'inputs': 13, 'oracles': ('selective_mmr_gold_type_oracle', 'combined_mmr_gte_gold_type_oracle'), 'primary': [50, 0.7]}


In [2]:
query_rows=read_csv(INPUTS['queries_13']); queries={}
for row in query_rows:
    if row['query_id'] in queries: assert queries[row['query_id']]=={'query_id':row['query_id'],'category':row['category'],'expected_card':row['expected_card']}
    else: queries[row['query_id']]={'query_id':row['query_id'],'category':row['category'],'expected_card':row['expected_card']}
assert len(queries)==30 and {category:sum(row['category']==category for row in queries.values()) for category in ('proper_noun','numeric_condition','semantic')}=={'proper_noun':10,'numeric_condition':10,'semantic':10}
SOURCE_WEIGHT='vector_0.4_bm25_0.6'; SOURCE_COMPARISON={20:'vector_0.4_bm25_0.6_top20',50:'vector_0.4_bm25_0.6_top50'}
rows19=read_csv(INPUTS['per_query_19']); lookup19={(row['configuration'],int(row['depth']),row['system'],row['query_id'],row['view']):row for row in rows19}; assert len(lookup19)==len(rows19)==1200
all_rows18=read_csv(INPUTS['per_query_18']); canonical_keys18=[(row['comparison'],row['weight'],int(row['depth']),row['system'],row['query_id'],row['view']) for row in all_rows18]; assert len(canonical_keys18)==len(set(canonical_keys18))==len(all_rows18)==1200
ambiguous_keys18=[(int(row['depth']),row['system'],row['query_id'],row['view']) for row in all_rows18]; ambiguous_collision_count=len(ambiguous_keys18)-len(set(ambiguous_keys18)); assert ambiguous_collision_count==600
lookup18={key:row for key,row in zip(canonical_keys18,all_rows18)}
def key18(depth,system,query_id,view): return (SOURCE_COMPARISON[depth],SOURCE_WEIGHT,depth,system,query_id,view)
rows18=[row for row in all_rows18 if row['comparison']==SOURCE_COMPARISON[int(row['depth'])] and row['weight']==SOURCE_WEIGHT]; assert len(rows18)==600
for depth in DEPTHS:
    for query_id in queries:
        for view in VIEWS:
            left=lookup19[(SOURCE_WEIGHT,depth,'no_mmr',query_id,view)]; right=lookup18[key18(depth,'no_reranker',query_id,view)]
            assert all(left[key]==right[key] for key in SOURCE_PAYLOAD_FIELDS) and source_payload_digest(left)==source_payload_digest(right)
selective18=[row for row in read_csv(INPUTS['selective_gte_per_query_18']) if row['comparison']==SOURCE_COMPARISON[int(row['depth'])] and row['weight']==SOURCE_WEIGHT]; selective18_keys=[(row['comparison'],row['weight'],int(row['depth']),row['system'],row['query_id'],row['view']) for row in selective18]; assert len(selective18_keys)==len(set(selective18_keys))==300
selective18_lookup={key:row for key,row in zip(selective18_keys,selective18)}
def selective18_key(depth,query_id,view): return (SOURCE_COMPARISON[depth],SOURCE_WEIGHT,depth,'gold_query_type_selective_oracle',query_id,view)
for key,row in selective18_lookup.items():
    depth,query_id,view=key[2],key[4],key[5]; source_system='gte' if queries[query_id]['category']=='semantic' else 'no_reranker'; source=lookup18[key18(depth,source_system,query_id,view)]
    assert all(row[key]==source[key] for key in ('top5_chunk_ids','top5_cards','top5_levels')) and all(abs(float(row[metric])-float(source[metric]))<=TOLERANCE for metric in METRICS)
trace=read_csv(INPUTS['trace_19']); assert len(trace)==6300
for lambda_value in LAMBDAS:
    for query_id in queries:
        top20=[row for row in trace if int(row['depth'])==20 and abs(float(row['lambda'])-lambda_value)<=TOLERANCE and row['query_id']==query_id]; top50=[row for row in trace if int(row['depth'])==50 and abs(float(row['lambda'])-lambda_value)<=TOLERANCE and row['query_id']==query_id]
        assert len(top20)==20 and len(top50)==50 and {row['chunk_id'] for row in top20}=={row['chunk_id'] for row in top50 if int(row['original_rank'])<=20}
        for depth,source in ((20,top20),(50,top50)):
            ordered=[row['chunk_id'] for row in sorted(source,key=lambda row:int(row['selection_step']))][:5]; metric_top5=json.loads(lookup19[(SOURCE_WEIGHT,depth,f'mmr_{lambda_value:.1f}',query_id,'strict_raw')]['top5_chunk_ids']); assert ordered==metric_top5
group_by_chunk={}
for row in read_jsonl(INPUTS['exact_groups_18']):
    for chunk_id in row['member_chunk_ids']: assert chunk_id not in group_by_chunk; group_by_chunk[chunk_id]=row['group_id']
redundancy19={(row['configuration'],int(row['depth']),row['system'],row['query_id']):row for row in read_csv(INPUTS['redundancy_19'])}; assert len(redundancy19)==240
source_audit={'canonical_18_key_fields':['comparison','weight','depth','system','query_id','view'],'canonical_18_keys_unique':True,'ambiguous_key_fields_prohibited':['depth','system','query_id','view'],'ambiguous_key_collision_count':ambiguous_collision_count,'canonical_no_mmr_payload_mismatch_count':0}
print({'source_rows':[len(rows18),len(rows19),len(selective18)],'canonical_no_mmr_payload_mismatches':0,'ambiguous_key_collision_count':ambiguous_collision_count,'top20_top50_candidate_contract':True})


{'source_rows': [600, 1200, 300], 'canonical_no_mmr_payload_mismatches': 0, 'ambiguous_key_collision_count': 600, 'top20_top50_candidate_contract': True}


In [3]:
def source_row(oracle,depth,lambda_value,query_id,view):
    category=queries[query_id]['category']
    if category=='numeric_condition': return lookup19[(SOURCE_WEIGHT,depth,f'mmr_{lambda_value:.1f}',query_id,view)],'19_mmr'
    if oracle=='combined_mmr_gte_gold_type_oracle' and category=='semantic': return lookup18[key18(depth,'gte',query_id,view)],'18_gte'
    return lookup19[(SOURCE_WEIGHT,depth,'no_mmr',query_id,view)],'19_no_mmr'
def source_identity(source,source_name,depth,query_id,view):
    if source_name=='18_gte': return {'source_schema':'18_per_query_metrics','source_comparison':SOURCE_COMPARISON[depth],'source_weight':SOURCE_WEIGHT,'source_configuration':SOURCE_WEIGHT,'source_depth':depth,'source_system':'gte','source_query_id':query_id,'source_view':view,'source_canonical_key':json.dumps(list(key18(depth,'gte',query_id,view)),separators=(',',':')),'source_row_digest':source_payload_digest(source)}
    system=source['system']; key=(SOURCE_WEIGHT,depth,system,query_id,view); return {'source_schema':'19_mmr_per_query_metrics','source_comparison':'','source_weight':SOURCE_WEIGHT,'source_configuration':SOURCE_WEIGHT,'source_depth':depth,'source_system':system,'source_query_id':query_id,'source_view':view,'source_canonical_key':json.dumps(list(key),separators=(',',':')),'source_row_digest':source_payload_digest(source)}
oracle_rows=[]
for oracle in ORACLES:
    for depth in DEPTHS:
        for lambda_value in LAMBDAS:
            for query_id in queries:
                for view in VIEWS:
                    source,source_name=source_row(oracle,depth,lambda_value,query_id,view); audit=source_identity(source,source_name,depth,query_id,view); row={'oracle':oracle,'configuration':SOURCE_WEIGHT,'depth':depth,'lambda':lambda_value,'query_id':query_id,'question_group':source['question_group'],'category':source['category'],'view':view,'selected_source':source_name,**audit,**{metric:float(source[metric]) for metric in METRICS},'relevant_unit_count':int(source['relevant_unit_count']),'ranking_unit_count':int(source['ranking_unit_count']),'top5_chunk_ids':source['top5_chunk_ids'],'top5_cards':source['top5_cards'],'top5_levels':source['top5_levels']}
                    assert all(row[key]==source[key] for key in ('top5_chunk_ids','top5_cards','top5_levels')) and all(abs(row[metric]-float(source[metric]))<=TOLERANCE for metric in METRICS)
                    assert row['source_row_digest']==source_payload_digest(source) and row['source_depth']==depth and row['source_query_id']==query_id and row['source_view']==view and row['source_weight']==SOURCE_WEIGHT
                    oracle_rows.append(row)
assert len(oracle_rows)==1800 and all(sum(row['oracle']==oracle for row in oracle_rows)==900 for oracle in ORACLES)
for oracle in ORACLES:
    for depth in DEPTHS:
        for lambda_value in LAMBDAS:
            for query_id in queries:
                same=[row for row in oracle_rows if row['oracle']==oracle and row['depth']==depth and row['lambda']==lambda_value and row['query_id']==query_id]; assert len(same)==5 and len({row['top5_chunk_ids'] for row in same})==1 and len({row['card_hit_at_3'] for row in same})==1
summary=[]
for oracle in ORACLES:
    for depth in DEPTHS:
        for lambda_value in LAMBDAS:
            for view in VIEWS:
                rows=[row for row in oracle_rows if row['oracle']==oracle and row['depth']==depth and row['lambda']==lambda_value and row['view']==view]
                for group in GROUPS:
                    selected=[row for row in rows if in_group(row,group)]; assert len(selected)==DENOMINATORS[group]
                    summary.append({'oracle':oracle,'configuration':'vector_0.4_bm25_0.6','depth':depth,'lambda':lambda_value,'view':view,'group':group,'denominator':len(selected),**{metric:statistics.fmean(row[metric] for row in selected) for metric in METRICS}})
assert len(summary)==300
primary_mrr=next(row['mrr_at_5'] for row in summary if row['oracle']=='combined_mmr_gte_gold_type_oracle' and row['depth']==50 and row['lambda']==0.7 and row['view']=='strict_raw' and row['group']=='evidence'); assert abs(primary_mrr-0.7083333333333333)<=TOLERANCE
print({'oracle_rows':len(oracle_rows),'summary':len(summary),'primary_combined_strict_evidence_mrr':primary_mrr})


{'oracle_rows': 1800, 'summary': 300, 'primary_combined_strict_evidence_mrr': 0.7083333333333333}


In [4]:
oracle_lookup={(row['oracle'],row['depth'],row['lambda'],row['query_id'],row['view']):row for row in oracle_rows}
def reference_row(name,depth,lambda_value,query_id,view):
    if name=='no_mmr': return lookup19[(SOURCE_WEIGHT,depth,'no_mmr',query_id,view)]
    if name=='same_lambda_all_mmr': return lookup19[(SOURCE_WEIGHT,depth,f'mmr_{lambda_value:.1f}',query_id,view)]
    if name=='all_gte': return lookup18[key18(depth,'gte',query_id,view)]
    if name=='selective_gte_gold_type_oracle': return selective18_lookup[selective18_key(depth,query_id,view)]
    return oracle_lookup[('selective_mmr_gold_type_oracle',depth,lambda_value,query_id,view)]
def reference_identity(name,row,depth,lambda_value,query_id,view):
    if name=='all_gte': return {'reference_source_schema':'18_per_query_metrics','reference_source_comparison':SOURCE_COMPARISON[depth],'reference_source_weight':SOURCE_WEIGHT,'reference_source_configuration':SOURCE_WEIGHT,'reference_source_depth':depth,'reference_source_system':'gte','reference_source_query_id':query_id,'reference_source_view':view,'reference_source_canonical_key':json.dumps(list(key18(depth,'gte',query_id,view)),separators=(',',':')),'reference_source_row_digest':source_payload_digest(row)}
    if name=='selective_gte_gold_type_oracle':
        key=selective18_key(depth,query_id,view); return {'reference_source_schema':'18_selective_oracle_per_query','reference_source_comparison':SOURCE_COMPARISON[depth],'reference_source_weight':SOURCE_WEIGHT,'reference_source_configuration':SOURCE_WEIGHT,'reference_source_depth':depth,'reference_source_system':'gold_query_type_selective_oracle','reference_source_query_id':query_id,'reference_source_view':view,'reference_source_canonical_key':json.dumps(list(key),separators=(',',':')),'reference_source_row_digest':source_payload_digest(row)}
    if name=='same_lambda_selective_mmr':
        key=('selective_mmr_gold_type_oracle',SOURCE_WEIGHT,depth,lambda_value,query_id,view); return {'reference_source_schema':'20_selective_mmr_oracle','reference_source_comparison':'','reference_source_weight':SOURCE_WEIGHT,'reference_source_configuration':SOURCE_WEIGHT,'reference_source_depth':depth,'reference_source_system':'selective_mmr_gold_type_oracle','reference_source_query_id':query_id,'reference_source_view':view,'reference_source_canonical_key':json.dumps(list(key),separators=(',',':')),'reference_source_row_digest':source_payload_digest(row)}
    system='no_mmr' if name=='no_mmr' else f'mmr_{lambda_value:.1f}'; key=(SOURCE_WEIGHT,depth,system,query_id,view); return {'reference_source_schema':'19_mmr_per_query_metrics','reference_source_comparison':'','reference_source_weight':SOURCE_WEIGHT,'reference_source_configuration':SOURCE_WEIGHT,'reference_source_depth':depth,'reference_source_system':system,'reference_source_query_id':query_id,'reference_source_view':view,'reference_source_canonical_key':json.dumps(list(key),separators=(',',':')),'reference_source_row_digest':source_payload_digest(row)}
paired=[]
for oracle in ORACLES:
    for depth in DEPTHS:
        for lambda_value in LAMBDAS:
            for reference in REFERENCES[oracle]:
                for query_id in queries:
                    for view in VIEWS:
                        current=oracle_lookup[(oracle,depth,lambda_value,query_id,view)]; base=reference_row(reference,depth,lambda_value,query_id,view); audit=reference_identity(reference,base,depth,lambda_value,query_id,view); row={'oracle':oracle,'depth':depth,'lambda':lambda_value,'reference':reference,'query_id':query_id,'question_group':current['question_group'],'category':current['category'],'view':view,**audit}
                        for metric in METRICS:
                            delta=current[metric]-float(base[metric]); row[f'oracle_{metric}']=current[metric]; row[f'reference_{metric}']=float(base[metric]); row[f'{metric}_delta']=delta; row[f'{metric}_outcome']='win' if delta>TOLERANCE else ('loss' if delta < -TOLERANCE else 'tie')
                        assert row['reference_source_row_digest']==source_payload_digest(base) and row['reference_source_weight']==SOURCE_WEIGHT and row['reference_source_depth']==depth and row['reference_source_query_id']==query_id and row['reference_source_view']==view
                        paired.append(row)
assert len(paired)==6300
wlt=[]
for oracle in ORACLES:
    for depth in DEPTHS:
        for lambda_value in LAMBDAS:
            for reference in REFERENCES[oracle]:
                for view in VIEWS:
                    for group in GROUPS:
                        rows=[row for row in paired if row['oracle']==oracle and row['depth']==depth and row['lambda']==lambda_value and row['reference']==reference and row['view']==view and in_group(row,group)]; assert len(rows)==DENOMINATORS[group]
                        for metric in METRICS:
                            counts={outcome:sum(row[f'{metric}_outcome']==outcome for row in rows) for outcome in ('win','loss','tie')}; mean_delta=statistics.fmean(row[f'{metric}_delta'] for row in rows); oracle_mean=statistics.fmean(row[f'oracle_{metric}'] for row in rows); reference_mean=statistics.fmean(row[f'reference_{metric}'] for row in rows); assert sum(counts.values())==len(rows) and abs(mean_delta-(oracle_mean-reference_mean))<=TOLERANCE
                            wlt.append({'oracle':oracle,'depth':depth,'lambda':lambda_value,'reference':reference,'view':view,'group':group,'metric':metric,'denominator':len(rows),'wins':counts['win'],'losses':counts['loss'],'ties':counts['tie'],'mean_delta':mean_delta})
assert len(wlt)==5250
structural=[]
for oracle in ORACLES:
    for depth in DEPTHS:
        for lambda_value in LAMBDAS:
            for query_id,query in queries.items():
                source=oracle_lookup[(oracle,depth,lambda_value,query_id,'strict_raw')]; ids=json.loads(source['top5_chunk_ids']); cards=json.loads(source['top5_cards']); levels=json.loads(source['top5_levels']); assert len(ids)==len(set(ids))==len(cards)==len(levels)==5 and all(chunk_id in group_by_chunk for chunk_id in ids)
                cross=sum(cards[i]==cards[j] and levels[i]!=levels[j] for i,j in itertools.combinations(range(5),2)); expected=sum(card==query['expected_card'] for card in cards); duplicate_excess=5-len({group_by_chunk[i] for i in ids})
                numeric_cosine=''
                if query['category']=='numeric_condition': numeric_cosine=float(redundancy19[(SOURCE_WEIGHT,depth,f'mmr_{lambda_value:.1f}',query_id)]['positive_pairwise_cosine_mean'])
                structural.append({'oracle':oracle,'depth':depth,'lambda':lambda_value,'query_id':query_id,'question_group':source['question_group'],'category':query['category'],'expected_card_count':expected,'off_card_count':5-expected,'expected_card_share_at_5':expected/5,'unique_card_count':len(set(cards)),'exact_duplicate_excess':duplicate_excess,'exact_unique_group_count':5-duplicate_excess,'same_card_cross_level_pair_count':cross,'numeric_mmr_positive_pairwise_cosine_mean':numeric_cosine,'cosine_scope':'numeric_mmr_only_n10' if query['category']=='numeric_condition' else 'N/A'} )
assert len(structural)==360 and all(row['numeric_mmr_positive_pairwise_cosine_mean']=='' for row in structural if row['category']!='numeric_condition')
structural_summary=[]
for oracle in ORACLES:
    for depth in DEPTHS:
        for lambda_value in LAMBDAS:
            for group in GROUPS:
                rows=[row for row in structural if row['oracle']==oracle and row['depth']==depth and row['lambda']==lambda_value and in_group(row,group)]; numeric=[row for row in rows if row['category']=='numeric_condition']; structural_summary.append({'oracle':oracle,'depth':depth,'lambda':lambda_value,'group':group,'denominator':len(rows),'expected_card_share_at_5':statistics.fmean(row['expected_card_share_at_5'] for row in rows),'unique_card_count':statistics.fmean(row['unique_card_count'] for row in rows),'exact_duplicate_excess':statistics.fmean(row['exact_duplicate_excess'] for row in rows),'same_card_cross_level_pair_count':statistics.fmean(row['same_card_cross_level_pair_count'] for row in rows),'positive_pairwise_cosine_mean':'' if len(numeric)!=10 else statistics.fmean(row['numeric_mmr_positive_pairwise_cosine_mean'] for row in numeric),'cosine_n':len(numeric) if len(numeric)==10 else 0,'cosine_scope':'numeric_mmr_only_n10' if len(numeric)==10 else 'N/A'})
assert len(structural_summary)==60
print({'paired':len(paired),'wlt':len(wlt),'structural':[len(structural),len(structural_summary)],'arithmetic_identity':True})


{'paired': 6300, 'wlt': 5250, 'structural': [360, 60], 'arithmetic_identity': True}


In [5]:
summary_lookup={(row['oracle'],row['depth'],row['lambda'],row['view'],row['group']):row for row in summary}
def ref_macro(reference,view,group,metric):
    rows=[reference_row(reference,PRIMARY_DEPTH,PRIMARY_LAMBDA,query_id,view) for query_id in queries if in_group({'question_group':'card' if queries[query_id]['category']=='proper_noun' else 'evidence','category':queries[query_id]['category']},group)]; assert len(rows)==DENOMINATORS[group]; return statistics.fmean(float(row[metric]) for row in rows)
combined=lambda view,group,metric:summary_lookup[('combined_mmr_gte_gold_type_oracle',50,0.7,view,group)][metric]
baseline=lambda view,group,metric:ref_macro('no_mmr',view,group,metric)
strict_nonreg={metric:combined('strict_raw','evidence',metric)+TOLERANCE>=baseline('strict_raw','evidence',metric) for metric in METRICS}
strict_improve=combined('strict_raw','evidence','mrr_at_5')>baseline('strict_raw','evidence','mrr_at_5')+TOLERANCE or combined('strict_raw','evidence','ndcg_at_5')>baseline('strict_raw','evidence','ndcg_at_5')+TOLERANCE
selective_noninferior={}
for reference in ('same_lambda_selective_mmr','selective_gte_gold_type_oracle'):
    for metric in ('mrr_at_5','ndcg_at_5'): selective_noninferior[f'{reference}_{metric}']=combined('strict_raw','evidence',metric)+TOLERANCE>=ref_macro(reference,'strict_raw','evidence',metric)
core_nonreg={}
for view,metrics in (('answer_bearing_raw',('hit_at_3','mrr_at_5')),('strict_exact_doc_dedup',('hit_at_3','mrr_at_5','ndcg_at_5')),('answer_bearing_exact_doc_dedup',('hit_at_3','mrr_at_5','ndcg_at_5')),('answer_bearing_gold_family_oracle',('hit_at_3','mrr_at_5','ndcg_at_5'))):
    for metric in metrics: core_nonreg[f'{view}_{metric}']=combined(view,'evidence',metric)+TOLERANCE>=baseline(view,'evidence',metric)
support=all(strict_nonreg.values()) and strict_improve and all(selective_noninferior.values()) and all(core_nonreg.values())
tradeoff=combined('strict_raw','evidence','mrr_at_5')>baseline('strict_raw','evidence','mrr_at_5')+TOLERANCE and any(not strict_nonreg[metric] for metric in ('card_hit_at_3','hit_at_3','recall_at_5','ndcg_at_5'))
best_selective={metric:max(ref_macro(reference,'strict_raw','evidence',metric) for reference in ('same_lambda_selective_mmr','selective_gte_gold_type_oracle')) for metric in ('mrr_at_5','ndcg_at_5')}
fails_both=combined('strict_raw','evidence','mrr_at_5')<=best_selective['mrr_at_5']+TOLERANCE and combined('strict_raw','evidence','ndcg_at_5')<=best_selective['ndcg_at_5']+TOLERANCE
diagnostic_label='counterfactual_complementarity_supported_on_dev' if support else ('metric_tradeoff_on_dev' if tradeoff else 'no_counterfactual_complementarity')
decision={'diagnostic_label':diagnostic_label,'support_gate_passed':support,'metric_tradeoff_triggered':tradeoff,'fails_to_exceed_best_selective_on_both_mrr_ndcg':fails_both,'checks':{'strict_nonregression':strict_nonreg,'strict_mrr_or_ndcg_improvement':strict_improve,'selective_noninferiority':selective_noninferior,'answer_exact_family_core_nonregression':core_nonreg},'primary':{'depth':50,'lambda':0.7,'combined_strict_evidence':{metric:combined('strict_raw','evidence',metric) for metric in METRICS},'baseline_strict_evidence':{metric:baseline('strict_raw','evidence',metric) for metric in METRICS},'best_selective_strict':best_selective},'final_status':['exploratory_gold_type_oracle_only','not_eligible_for_promotion','not_an_operational_router']}
summary_json={'schema_version':'selective_mmr_gte_oracle_summary_v2_canonical_source_keys','contract':CONTRACT,'source_audit':source_audit,'decision':decision,'primary_rows':[row for row in summary if row['depth']==50 and row['lambda']==0.7 and row['group'] in ('evidence','numeric','semantic')],'structural_summary':structural_summary,'row_counts':{'per_query':len(oracle_rows),'summary':len(summary),'paired':len(paired),'wlt':len(wlt),'structural':len(structural),'structural_summary':len(structural_summary)},'limitations':['gold query type is unavailable to an operational router','primary lambda was chosen adaptively after experiment 19','stored source rows bound all counterfactuals','combined whole-query cosine is N/A because GTE cosine was not stored','single development-set diagnostic only']}
print({'diagnostic_label':diagnostic_label,'support':support,'tradeoff':tradeoff,'primary_strict':decision['primary']['combined_strict_evidence']})


{'diagnostic_label': 'metric_tradeoff_on_dev', 'support': False, 'tradeoff': True, 'primary_strict': {'card_hit_at_3': 0.95, 'hit_at_3': 0.8, 'recall_at_5': 0.7, 'mrr_at_5': 0.7083333333333333, 'ndcg_at_5': 0.6254906244967336}}


In [6]:
write_csv(OUTPUT_ROOT/'oracle_per_query.csv',oracle_rows); write_csv(OUTPUT_ROOT/'oracle_summary.csv',summary); write_json(OUTPUT_ROOT/'oracle_summary.json',summary_json); write_csv(OUTPUT_ROOT/'oracle_paired_deltas.csv',paired); write_csv(OUTPUT_ROOT/'oracle_wlt.csv',wlt); write_csv(OUTPUT_ROOT/'oracle_structural_diagnostics.csv',structural); write_json(OUTPUT_ROOT/'oracle_decision.json',decision)
readme=f'''# 20 Selective MMR/GTE gold-type oracle

Codex coder agent가 `skn25` CPU/offline fresh kernel에서 실행한 개발셋 사후 진단이다. 정답 query type을 미리 아는 oracle이므로 실제 router가 아니다.

- selective MMR: 숫자형 질의만 MMR, 의미형·고유명사형은 no-MMR
- combined: 숫자형은 MMR, 의미형은 GTE, 고유명사형은 no-MMR
- paired delta/WLT: 같은 질의에서 기준 대비 지표 차이와 승/패/동률
- exact duplicate: 동일 카드와 동일 정규화 문서 group의 Top5 초과 중복
- combined 전체 cosine은 GTE cosine 미저장으로 N/A이며 numeric MMR 10질의만 별도 보고
- primary Top50 λ0.7은 19번 결과 후 정한 adaptive exploratory 조건
- diagnostic label: `{diagnostic_label}`
- final status: exploratory_gold_type_oracle_only / not_eligible_for_promotion / not_an_operational_router

저장된 source metric/Top5 행만 선택했으며 새 ranking·score 혼합·검색·재평가를 하지 않았다. 개발셋 단일 결과로 holdout·운영 일반화를 주장하지 않는다. GPU/model/custom code/network/API/new embedding/Chroma/package install은 모두 0이다.
'''
(OUTPUT_ROOT/'README.md').write_text(readme,encoding='utf-8')
input_hashes_after={name:sha(path) for name,path in INPUTS.items()}; assert input_hashes_after==input_hashes_before
output_names=('oracle_contract.json','oracle_per_query.csv','oracle_summary.csv','oracle_summary.json','oracle_paired_deltas.csv','oracle_wlt.csv','oracle_structural_diagnostics.csv','oracle_decision.json','README.md')
integrity={'schema_version':'selective_mmr_gte_oracle_integrity_v2_canonical_source_keys','self_hash_excluded':True,'inputs':{name:{'path':str(path.relative_to(PROJECT_ROOT)),'sha256_before':input_hashes_before[name],'sha256_after':input_hashes_after[name],'unchanged':True} for name,path in INPUTS.items()},'outputs':{name:{'sha256':sha(OUTPUT_ROOT/name),'bytes':(OUTPUT_ROOT/name).stat().st_size} for name in output_names},'notebook':{'path':'notebooks/20_selective_mmr_gte_gold_type_oracle.ipynb','sha256':'pending_after_nbclient_serialization'},'row_counts':summary_json['row_counts'],'source_audit':source_audit,'assertions':{'oracle_900_each_1800_total':True,'canonical_source_key_uniqueness':True,'ambiguous_key_collision_detected_and_prohibited':True,'canonical_no_mmr_payload_mismatch_count':0,'canonical_oracle_source_payload_mismatch_count':0,'canonical_all_gte_reference_payload_mismatch_count':0,'top20_top50_candidate_contract':True,'arithmetic_identity':True,'primary_mrr_exact':True,'structural_ties_and_wlt_denominators':True,'metric_range_card_view_invariant':True,'input_hashes_unchanged':True},'execution':CONTRACT['execution']}
write_json(OUTPUT_ROOT/'oracle_integrity.json',integrity)
expected={'oracle_per_query.csv':1800,'oracle_summary.csv':300,'oracle_paired_deltas.csv':6300,'oracle_wlt.csv':5250,'oracle_structural_diagnostics.csv':360}
for name,count in expected.items(): assert len(read_csv(OUTPUT_ROOT/name))==count
stored_oracle=read_csv(OUTPUT_ROOT/'oracle_per_query.csv'); stored_paired=read_csv(OUTPUT_ROOT/'oracle_paired_deltas.csv')
assert all(math.isfinite(float(row[metric])) and 0<=float(row[metric])<=1 for row in stored_oracle for metric in METRICS)
assert all(int(row['wins'])+int(row['losses'])+int(row['ties'])==int(row['denominator']) for row in read_csv(OUTPUT_ROOT/'oracle_wlt.csv'))
canonical_oracle_source_failures=0
for row in stored_oracle:
    key=tuple(json.loads(row['source_canonical_key'])); source=lookup18[(key[0],key[1],int(key[2]),key[3],key[4],key[5])] if row['source_schema']=='18_per_query_metrics' else lookup19[(key[0],int(key[1]),key[2],key[3],key[4])]
    canonical_oracle_source_failures+=int(row['source_row_digest']!=source_payload_digest(source) or any(row[field]!=source[field] for field in ('top5_chunk_ids','top5_cards','top5_levels')) or int(row['ranking_unit_count'])!=int(source['ranking_unit_count']) or int(row['relevant_unit_count'])!=int(source['relevant_unit_count']) or any(abs(float(row[metric])-float(source[metric]))>TOLERANCE for metric in METRICS))
assert canonical_oracle_source_failures==0
canonical_all_gte_reference_failures=0
for row in (row for row in stored_paired if row['reference']=='all_gte'):
    key=tuple(json.loads(row['reference_source_canonical_key'])); source=lookup18[(key[0],key[1],int(key[2]),key[3],key[4],key[5])]
    canonical_all_gte_reference_failures+=int(row['reference_source_row_digest']!=source_payload_digest(source) or any(abs(float(row[f'reference_{metric}'])-float(source[metric]))>TOLERANCE for metric in METRICS))
assert canonical_all_gte_reference_failures==0
assert {name:sha(path) for name,path in INPUTS.items()}==input_hashes_before
print({'validation':'PASS','rows':expected,'diagnostic_label':diagnostic_label,'canonical_failures':{'no_mmr_18_19':0,'oracle_sources':canonical_oracle_source_failures,'all_gte_references':canonical_all_gte_reference_failures},'ambiguous_key_collision_count':ambiguous_collision_count,'inputs_unchanged':True,'gpu_model_network_chroma_calls':0})


{'validation': 'PASS', 'rows': {'oracle_per_query.csv': 1800, 'oracle_summary.csv': 300, 'oracle_paired_deltas.csv': 6300, 'oracle_wlt.csv': 5250, 'oracle_structural_diagnostics.csv': 360}, 'diagnostic_label': 'metric_tradeoff_on_dev', 'canonical_failures': {'no_mmr_18_19': 0, 'oracle_sources': 0, 'all_gte_references': 0}, 'ambiguous_key_collision_count': 600, 'inputs_unchanged': True, 'gpu_model_network_chroma_calls': 0}
